# 04 — Infinite Slope Stability

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Katowice &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

The **infinite-slope model** is the simplest mechanical model of a translational landslide. It assumes a planar slip surface parallel to the ground, soil that is uniform along the slope, and end effects that can be neglected. Despite its simplicity, it is the work-horse of shallow-landslide hazard assessment — embedded in every regional susceptibility model from SHALSTAB to SLIDE — because it captures the essential physics: weight drives, friction and cohesion resist, water erodes the resistance by raising pore pressure.


## About this notebook

**Learning objectives.** By the end of this notebook the student will be able to:

1. State the assumptions of the infinite-slope model and recognise when they break down.
2. Derive and apply the factor-of-safety equation for an infinite slope with pore-water pressure.
3. Quantify the effect of saturation (parameter $m$) on slope stability.
4. Estimate a **critical failure depth** for a cohesive frictional soil.
5. Apply the model to the same Carpathian flysch slope from notebook 03 and reproduce the rainfall-triggered failure.

**Prerequisites.** Notebook `03 — Mohr circles & Mohr–Coulomb`. The lecture on *Soil mechanics and slope stability*.

> **For your PowerPoint deck.** Four SVG figures land in `figures/`:
> - `infinite_slope_geometry.svg` — labelled cross-section
> - `fs_vs_slope_angle.svg` — the classical FS-vs-β curve family
> - `fs_vs_saturation.svg` — the "drowning slope" curve
> - `fs_heatmap_beta_m.svg` — heatmap with the FS = 1 isoline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Arc

from style import apply_style, COLORS, save_figure
apply_style()

from ipywidgets import interact, FloatSlider

GAMMA_W = 9.81  # unit weight of water [kN/m^3]


## 1. Geometry and the FS equation

Consider a soil layer of thickness $z$ (measured perpendicular to the slope) resting on a planar slip surface inclined at angle $\beta$. The soil has unit weight $\gamma$, effective cohesion $c'$, and effective friction angle $\varphi'$. A water table sits at depth $z_w$ below the surface, so the **saturated fraction** of the layer is

$$
m \;=\; \frac{z - z_w}{z} \;\in\; [0,1].
\qquad (1)
$$

Assuming flow parallel to the slope, the stresses on the slip surface are:

$$
\sigma_n \;=\; \gamma\, z \cos^2\beta
\qquad (2)
$$

$$
\tau \;=\; \gamma\, z \sin\beta \cos\beta
\qquad (3)
$$

$$
u \;=\; m\, \gamma_w\, z \cos^2\beta
\qquad (4)
$$

Subtracting $u$ from $\sigma_n$ gives the effective normal stress, and applying the Mohr–Coulomb criterion as available shear strength gives the **factor of safety**:

$$
\boxed{\;
\mathrm{FS} \;=\; \frac{c'}{\gamma\, z \sin\beta \cos\beta}
\;+\;
\left(1 - m\,\frac{\gamma_w}{\gamma}\right)\frac{\tan\varphi'}{\tan\beta}.
\;}
\qquad (5)
$$

Three things are worth memorising from this equation:

- Cohesion enters as $c' / z$ — its stabilising effect **vanishes at depth**. Deep slip surfaces are governed by friction alone.
- The friction term is reduced by the saturation factor $(1 - m\,\gamma_w/\gamma)$. Full saturation ($m=1$) almost halves it.
- For a **cohesionless dry** slope, equation (5) collapses to $\mathrm{FS} = \tan\varphi'/\tan\beta$ — the classical *angle-of-repose* result.


In [ ]:
def fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m):
    """Infinite-slope factor of safety, equation (5)."""
    beta = np.radians(beta_deg)
    phi  = np.radians(phi_deg)
    cohesion_term = c_prime / (gamma * z * np.sin(beta) * np.cos(beta))
    friction_term = (1.0 - m * GAMMA_W / gamma) * np.tan(phi) / np.tan(beta)
    return cohesion_term + friction_term


def plot_slope_geometry(beta_deg=28.0, z=2.0, m=0.5):
    """Schematic cross-section of the infinite-slope geometry."""
    beta = np.radians(beta_deg)
    L = 9.0  # horizontal extent
    # Ground surface from (0, L tan b) down to (L, 0)
    x_surf = np.array([0.0, L])
    y_surf = np.array([L * np.tan(beta), 0.0])
    # Slip surface, offset perpendicular by z
    dx, dy = -z * np.sin(beta), -z * np.cos(beta)
    x_slip = x_surf + dx
    y_slip = y_surf + dy
    # Water-table surface, offset perpendicular by (1-m)*z
    zw = (1.0 - m) * z
    x_wt = x_surf - zw * np.sin(beta)
    y_wt = y_surf - zw * np.cos(beta)

    fig, ax = plt.subplots(figsize=(8.0, 4.6))

    # Soil polygon
    poly_x = np.concatenate([x_surf, x_slip[::-1]])
    poly_y = np.concatenate([y_surf, y_slip[::-1]])
    ax.fill(poly_x, poly_y, color=COLORS["soil"], alpha=0.25)

    # Saturated zone polygon (between water table and slip surface)
    wet_x = np.concatenate([x_wt, x_slip[::-1]])
    wet_y = np.concatenate([y_wt, y_slip[::-1]])
    ax.fill(wet_x, wet_y, color=COLORS["water"], alpha=0.25)

    # Surfaces
    ax.plot(x_surf, y_surf, color=COLORS["soil"], lw=2.0, label="ground surface")
    ax.plot(x_slip, y_slip, color=COLORS["fail"], lw=2.0, ls="--", label="slip surface")
    ax.plot(x_wt,   y_wt,   color=COLORS["water"], lw=1.5, ls=":", label="water table")

    # Horizontal reference and angle arc
    ax.plot([0, L], [0, 0], color="black", lw=0.6)
    arc = Arc((L, 0), 2.4, 2.4, angle=0,
              theta1=180 - beta_deg, theta2=180, color="black", lw=1.1)
    ax.add_patch(arc)
    ax.text(L - 1.6, 0.35, fr"$\beta = {beta_deg:.0f}^\circ$", fontsize=14)

    # z arrow (perpendicular to slope, near the middle)
    mid = 0.55
    ps = np.array([x_surf[0] + mid * (x_surf[1] - x_surf[0]),
                   y_surf[0] + mid * (y_surf[1] - y_surf[0])])
    pe = ps + np.array([dx, dy])
    ax.annotate("", xy=pe, xytext=ps,
                arrowprops=dict(arrowstyle="<->", color="black", lw=1.2))
    label_pt = 0.5 * (ps + pe) + np.array([-0.45, 0.10])
    ax.text(*label_pt, fr"$z = {z:.1f}$ m", fontsize=12)

    # m label inside the saturated wedge
    if m > 0.05:
        cx, cy = 0.65 * x_slip.mean() + 0.35 * x_wt.mean(), 0.65 * y_slip.mean() + 0.35 * y_wt.mean()
        ax.text(cx, cy - 0.3, fr"saturated:  $m = {m:.2f}$",
                fontsize=12, color=COLORS["water"])

    ax.set_xlim(-1.5, L + 0.5)
    ax.set_ylim(-0.5, y_surf[0] + 0.8)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.spines["bottom"].set_visible(False)
    ax.grid(False)
    ax.legend(loc="upper right", fontsize=11)
    ax.set_title("Infinite-slope geometry")
    return fig


fig = plot_slope_geometry(beta_deg=28, z=2.0, m=0.5)
save_figure(fig, "infinite_slope_geometry")
plt.show()


## 2. Dry slope — the angle-of-repose limit

With $c' = 0$ and $m = 0$ equation (5) reduces to

$$
\mathrm{FS}_{\text{dry, cohesionless}} \;=\; \frac{\tan\varphi'}{\tan\beta}.
\qquad (6)
$$

This is the foundational result of granular slope mechanics: a dry pile of cohesionless soil is stable while its slope is shallower than the friction angle, and at incipient failure exactly *when* $\beta = \varphi'$. The curve family below shows how cohesion lifts the FS curve, especially at depth-limited shallow slip surfaces.


In [ ]:
beta_grid = np.linspace(10, 50, 200)
gamma = 19.0   # kN/m^3, Carpathian flysch silty sand
z = 2.0        # m

fig, ax = plt.subplots()
for c_prime, ls in [(0.0, "-"), (5.0, "--"), (15.0, ":")]:
    for phi_deg, colour in [(25.0, COLORS["neutral"]),
                            (32.0, COLORS["accent"]),
                            (38.0, COLORS["safe"])]:
        fs = [fs_infinite_slope(b, z, c_prime, phi_deg, gamma, m=0.0) for b in beta_grid]
        label = (fr"$\varphi'$ = {phi_deg:.0f}$^\circ$, $c'$ = {c_prime:.0f} kPa"
                 if ls == "-" else None)
        ax.plot(beta_grid, fs, color=colour, ls=ls, label=label)

ax.axhline(1.0, color=COLORS["fail"], lw=1.2, ls="-")
ax.text(48, 1.05, "FS = 1", color=COLORS["fail"], fontsize=11, ha="right")
ax.set_ylim(0.0, 3.5)
ax.set_xlabel(r"slope angle  $\beta$  [$^\circ$]")
ax.set_ylabel("factor of safety  FS")
ax.set_title(fr"Dry slope, $z$ = {z:.1f} m, $\gamma$ = {gamma:.0f} kN/m$^3$  "
             r"(solid = $c'$ = 0;  dashed = 5 kPa;  dotted = 15 kPa)")
ax.legend(loc="upper right", fontsize=11)
save_figure(fig, "fs_vs_slope_angle")
plt.show()


## 3. The saturation factor $m$

Even modest amounts of saturation can drop a marginally stable slope below FS = 1. With $\gamma \approx 19$ kN/m³ and $\gamma_w \approx 9.81$ kN/m³, full saturation ($m = 1$) shaves about **52 %** off the friction contribution. That is the mechanical reason why rainfall-triggered shallow slides cluster around storms that *raise the water table to (or near) the ground surface*, not around storms that merely deliver a lot of cumulative water (Iverson, 2000).

The curve below tracks FS as the saturated fraction $m$ grows from a dry slope to fully drowned, for a Carpathian flysch slope at $\beta = 28^\circ$.


In [ ]:
beta_deg = 28.0
phi_deg = 28.0
gamma   = 19.0
z       = 2.0
m_grid  = np.linspace(0.0, 1.0, 200)

fig, ax = plt.subplots()
for c_prime, colour in [(0.0,  COLORS["neutral"]),
                        (5.0,  COLORS["accent"]),
                        (15.0, COLORS["safe"])]:
    fs = [fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m) for m in m_grid]
    ax.plot(m_grid, fs, color=colour, lw=2.0,
            label=fr"$c'$ = {c_prime:.0f} kPa")

ax.axhline(1.0, color=COLORS["fail"], lw=1.2)
ax.text(0.02, 1.05, "FS = 1", color=COLORS["fail"], fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(0.0, 2.5)
ax.set_xlabel(r"saturation fraction  $m = (z - z_w) / z$")
ax.set_ylabel("factor of safety  FS")
ax.set_title(fr"Drowning a slope:  $\beta$ = {beta_deg:.0f}$^\circ$,  "
             fr"$\varphi'$ = {phi_deg:.0f}$^\circ$,  $z$ = {z:.1f} m")
ax.legend(loc="upper right")
save_figure(fig, "fs_vs_saturation")
plt.show()


## 4. Combined sensitivity: $\mathrm{FS}(\beta, m)$

The two previous plots are slices through a 2-D surface. Here is the surface itself — a heatmap of FS over slope angle and saturation, with the FS = 1 contour marked. For a Carpathian flysch soil ($c'$ = 5 kPa, $\varphi'$ = 28°, $z$ = 2 m, $\gamma$ = 19 kN/m³), the boundary between safe and unsafe traces the **diagonal corridor** along which most shallow translational slides in this terrain initiate.


In [ ]:
c_prime = 5.0
phi_deg = 28.0
gamma   = 19.0
z       = 2.0

beta_axis = np.linspace(10, 45, 160)
m_axis    = np.linspace(0.0, 1.0, 140)
B, M = np.meshgrid(beta_axis, m_axis)
FS = np.vectorize(fs_infinite_slope)(B, z, c_prime, phi_deg, gamma, M)

fig, ax = plt.subplots(figsize=(8.0, 5.5))
levels = np.linspace(0.4, 2.2, 19)
cf = ax.contourf(B, M, FS, levels=levels, cmap="RdYlGn", extend="both")
cs = ax.contour(B, M, FS, levels=[1.0], colors="black", linewidths=2.0)
ax.clabel(cs, fmt="FS = 1", inline=True, fontsize=11)

cb = fig.colorbar(cf, ax=ax, shrink=0.92)
cb.set_label("factor of safety  FS")

ax.set_xlabel(r"slope angle  $\beta$  [$^\circ$]")
ax.set_ylabel(r"saturation fraction  $m$")
ax.set_title(fr"FS heatmap — Carpathian flysch  "
             fr"($c'$ = {c_prime:.0f} kPa, $\varphi'$ = {phi_deg:.0f}$^\circ$, "
             fr"$z$ = {z:.1f} m)")
save_figure(fig, "fs_heatmap_beta_m")
plt.show()


## 5. Interactive exploration

Slide the parameters and watch FS update. A few sanity checks worth doing yourself:

- Set $c' = 0$ and $m = 0$. At what $\beta$ does the slope sit on the FS = 1 line? Compare to $\varphi'$.
- Hold $\beta$ just below $\varphi'$ with $c' = 5$ kPa. How small does $m$ need to be to keep FS above 1?
- Increase $z$ from 0.5 m to 5 m. Note how the cohesion contribution fades — the slope behaves like a cohesionless granular pile at depth.


In [ ]:
def explore(beta_deg=28.0, z=2.0, c_prime=5.0, phi_deg=28.0,
            gamma=19.0, m=0.3):
    fs = fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m)
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6),
                             gridspec_kw=dict(width_ratios=[1.2, 1.0]))
    # Left: geometry sketch
    plt.sca(axes[0])
    # Re-using plot_slope_geometry would create its own fig; redraw inline:
    beta = np.radians(beta_deg)
    L = 9.0
    x_surf = np.array([0.0, L]); y_surf = np.array([L*np.tan(beta), 0.0])
    dx, dy = -z*np.sin(beta), -z*np.cos(beta)
    x_slip = x_surf + dx; y_slip = y_surf + dy
    zw = (1.0 - m) * z
    x_wt = x_surf - zw*np.sin(beta); y_wt = y_surf - zw*np.cos(beta)
    axes[0].fill(np.concatenate([x_surf, x_slip[::-1]]),
                 np.concatenate([y_surf, y_slip[::-1]]),
                 color=COLORS["soil"], alpha=0.25)
    axes[0].fill(np.concatenate([x_wt, x_slip[::-1]]),
                 np.concatenate([y_wt, y_slip[::-1]]),
                 color=COLORS["water"], alpha=0.25)
    axes[0].plot(x_surf, y_surf, color=COLORS["soil"], lw=2.0)
    axes[0].plot(x_slip, y_slip, color=COLORS["fail"], lw=2.0, ls="--")
    axes[0].plot(x_wt, y_wt, color=COLORS["water"], lw=1.5, ls=":")
    axes[0].set_xlim(-1.5, L + 0.5)
    axes[0].set_ylim(-0.5, max(y_surf[0] + 0.8, 4.0))
    axes[0].set_aspect("equal"); axes[0].set_xticks([]); axes[0].set_yticks([])
    axes[0].spines["left"].set_visible(False); axes[0].spines["bottom"].set_visible(False)
    axes[0].grid(False)
    axes[0].set_title(fr"$\beta$ = {beta_deg:.0f}$^\circ$,  z = {z:.1f} m,  m = {m:.2f}")

    # Right: FS vs m curve for the chosen geometry
    m_grid = np.linspace(0, 1, 200)
    fs_grid = [fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, mm) for mm in m_grid]
    axes[1].plot(m_grid, fs_grid, color=COLORS["accent"], lw=2.0)
    axes[1].axhline(1.0, color=COLORS["fail"], lw=1.0)
    axes[1].plot([m], [fs], "o", ms=10,
                 color=(COLORS["safe"] if fs > 1 else COLORS["fail"]))
    axes[1].set_xlim(0, 1); axes[1].set_ylim(0, max(2.5, fs * 1.3))
    axes[1].set_xlabel(r"$m$")
    axes[1].set_ylabel("FS")
    axes[1].set_title(fr"FS = {fs:.2f}")
    plt.show()


interact(
    explore,
    beta_deg=FloatSlider(min=10, max=45, step=1, value=28.0, description=r"$\beta$ [deg]"),
    z       =FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description="z [m]"),
    c_prime =FloatSlider(min=0, max=30, step=1, value=5.0, description=r"$c'$ [kPa]"),
    phi_deg =FloatSlider(min=15, max=42, step=1, value=28.0, description=r"$\varphi'$ [deg]"),
    gamma   =FloatSlider(min=15, max=22, step=0.5, value=19.0, description=r"$\gamma$ [kN/m$^3$]"),
    m       =FloatSlider(min=0.0, max=1.0, step=0.05, value=0.3, description="m"),
);


## 6. Worked example: same Carpathian flysch slope, rainfall trigger

We continue the slope from notebook 03:

|             | value      |
|-------------|------------|
| $\beta$     | 28°        |
| $z$         | 2.0 m      |
| $c'$        | 5 kPa      |
| $\varphi'$  | 28°        |
| $\gamma$    | 19 kN/m³   |

Note that $\beta = \varphi'$: without cohesion this slope would sit *exactly* on the angle of repose. The 5 kPa of cohesion is all that keeps it standing.

Before the storm, the water table sits well below the slip surface ($m \approx 0.3$). After three days of heavy rain, a perched water table climbs almost to the surface ($m \approx 0.9$). What does the infinite-slope FS do?


In [ ]:
beta_deg, z, c_prime, phi_deg, gamma = 28.0, 2.0, 5.0, 28.0, 19.0

m_pre, m_post = 0.30, 0.90
FS_dry  = fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m=0.0)
FS_pre  = fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m_pre)
FS_post = fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m_post)

print(f"Fully dry slope (m = 0.00):  FS = {FS_dry:.2f}")
print(f"Before rain    (m = {m_pre:.2f}):  FS = {FS_pre:.2f}")
print(f"After rain     (m = {m_post:.2f}):  FS = {FS_post:.2f}")

# Plot the FS-vs-m curve with the three states marked.
m_grid  = np.linspace(0, 1, 200)
fs_grid = [fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, mm)
           for mm in m_grid]

fig, ax = plt.subplots()
ax.plot(m_grid, fs_grid, color=COLORS["accent"], lw=2.2)
ax.axhline(1.0, color=COLORS["fail"], lw=1.2)
ax.text(0.02, 1.05, "FS = 1", color=COLORS["fail"], fontsize=11)

for m_val, fs_val, label in [(0.0,    FS_dry,  "fully dry"),
                             (m_pre,  FS_pre,  "before rain"),
                             (m_post, FS_post, "after rain")]:
    colour = COLORS["safe"] if fs_val > 1.0 else COLORS["fail"]
    ax.plot([m_val], [fs_val], "o", ms=10, color=colour)
    ax.annotate(fr"{label}, FS = {fs_val:.2f}",
                xy=(m_val, fs_val), xytext=(8, 12), textcoords="offset points",
                fontsize=11, color=colour)

ax.set_xlim(0, 1); ax.set_ylim(0.0, 2.5)
ax.set_xlabel(r"saturation fraction  $m$")
ax.set_ylabel("factor of safety  FS")
ax.set_title(fr"Carpathian flysch slope:  $\beta$ = {beta_deg:.0f}$^\circ$ = $\varphi'$,  "
             fr"$c'$ = {c_prime:.0f} kPa,  $z$ = {z:.1f} m")
save_figure(fig, "worked_example_carpathian")
plt.show()


### Critical failure depth

For $c' > 0$, the FS equation has a peculiar property: the cohesion term grows without bound as $z \to 0$. This means very thin soil layers are *more* stable than thicker ones, all else equal. Setting $\mathrm{FS} = 1$ in equation (5) and solving for $z$ gives the **critical depth** at which the slip surface first becomes unstable for a given $m$:

$$
z_{\text{crit}}(m) \;=\; \frac{c'}{\gamma \sin\beta \cos\beta \,\bigl[\,1 - \bigl(1 - m\,\gamma_w/\gamma\bigr)\tan\varphi'/\tan\beta\,\bigr]}.
\qquad (7)
$$

This is the depth around which shallow translational slides preferentially nucleate. The plot below traces $z_{\text{crit}}$ as the water table rises.


In [ ]:
def z_critical(beta_deg, c_prime, phi_deg, gamma, m):
    """Depth at which FS = 1 for given (beta, c', phi', gamma, m), eq. (7)."""
    beta = np.radians(beta_deg)
    phi  = np.radians(phi_deg)
    bracket = 1.0 - (1.0 - m * GAMMA_W / gamma) * np.tan(phi) / np.tan(beta)
    if bracket <= 0:
        return np.inf  # cohesion-only stability; cannot reach FS=1 at any depth
    return c_prime / (gamma * np.sin(beta) * np.cos(beta) * bracket)


m_grid = np.linspace(0.0, 1.0, 200)
z_crit = np.array([z_critical(beta_deg, c_prime, phi_deg, gamma, m) for m in m_grid])

fig, ax = plt.subplots()
mask = np.isfinite(z_crit) & (z_crit < 10)
ax.plot(m_grid[mask], z_crit[mask], color=COLORS["fail"], lw=2.2)
ax.fill_between(m_grid[mask], 0, z_crit[mask], color=COLORS["safe"], alpha=0.18,
                label="stable (above this depth FS > 1)")
ax.fill_between(m_grid[mask], z_crit[mask], 10, color=COLORS["fail"], alpha=0.15,
                label="unstable (below this depth FS < 1)")
ax.set_ylim(0, 6)
ax.set_xlim(0, 1)
ax.invert_yaxis()  # depth grows downward
ax.set_xlabel(r"saturation fraction  $m$")
ax.set_ylabel(r"critical depth  $z_{\mathrm{crit}}$  [m]")
ax.set_title(fr"Critical failure depth, Carpathian flysch  "
             fr"($\beta$ = {beta_deg:.0f}$^\circ$, $c'$ = {c_prime:.0f} kPa)")
ax.legend(loc="lower left")
save_figure(fig, "critical_depth")
plt.show()


## 7. When the infinite-slope model breaks down

The model assumes a *long, uniform* slope with a *planar slip surface* parallel to the ground. Three situations where this fails badly:

- **Deep-seated rotational slides.** The slip surface is curved, end effects matter, and FS must be computed by limit-equilibrium methods (Bishop, Janbu, Spencer) or numerical methods. Use Bishop's simplified method or finite-element strength-reduction analysis instead.
- **Strongly layered or heterogeneous soils.** Strength varies along the slip plane and the assumption of single $(c', \varphi')$ becomes meaningless. Use a method-of-slices approach that lets each slice carry its own properties.
- **Short slopes with significant lateral/end resistance.** Real failures of a few metres width recruit cohesion and friction along their lateral margins. The infinite-slope FS is a *lower bound* and may be conservative by a factor of 2–4.

Despite this, the infinite-slope model remains the right tool for **regional shallow-landslide susceptibility**, where you have noisy parameter estimates over thousands of km² and just want a defensible first-order map.


## Take-aways

- Infinite-slope FS is a one-equation summary of the mechanics: cohesion / weight in the numerator, weight / shear in the denominator, with water doing its damage by *reducing the friction contribution* via $(1 - m \gamma_w/\gamma)$.
- Cohesion stabilises shallow slides; below the **critical depth** the slope is at the mercy of friction alone.
- Saturation cuts the friction term by up to ~50% — this is *the* mechanical reason rainfall triggers shallow translational slides.
- The model is the work-horse of regional susceptibility mapping but should be replaced by limit-equilibrium or finite-element methods for site-specific stability analysis of deep, layered, or short slopes.


## Questions for the exam

1. Derive equation (5) starting from equations (2)–(4) and the Mohr–Coulomb criterion. State each assumption as you use it.
2. A Tatra debris cover has $c' = 0$, $\varphi' = 34^\circ$, $\gamma = 18$ kN/m³, $z = 1.2$ m, slope $\beta = 30^\circ$. Compute the dry FS and the fully saturated FS. By what fraction does saturation reduce the FS?
3. Using equation (7), compute the critical depth for a slope with $\beta = 30^\circ$, $c' = 8$ kPa, $\varphi' = 30^\circ$, $\gamma = 19$ kN/m³, fully saturated ($m = 1$). Below what depth would you expect shallow translational slides to nucleate?
4. The infinite-slope model is widely used in regional susceptibility mapping but rarely used to assess a specific landslide once it has happened. Why?


## References

- Iverson, R. M. (2000). *Landslide triggering by rain infiltration.* Water Resources Research, 36(7), 1897–1910.
- Hungr, O., Leroueil, S., & Picarelli, L. (2014). *The Varnes classification of landslide types, an update.* Landslides, 11(2), 167–194. https://doi.org/10.1007/s10346-013-0436-y
- Skempton, A. W. & DeLory, F. A. (1957). *Stability of natural slopes in London Clay.* Proceedings of the 4th International Conference on Soil Mechanics and Foundation Engineering, 2, 378–381. *(The original infinite-slope analysis.)*
- Tichavský, R., et al. (2019). *Dry spells and extreme precipitation are the main trigger of landslides in Central Europe.* Sci. Reports, 9, 14560.
